# Validate TD and FD→TD waveform-mode paths

This notebook compares waveform modes generated with consistent physical/numerical parameters, except for the requested starting-frequency convention:

- `NRSur7dq4`: `f_lower = 0.0`;
- `IMRPhenomXPHM`: `f_lower = 10.0`.

All remaining shared parameters are taken from the same `COMMON_PARAMS` dictionary. The notebook also compares the different FD→TD implementation routes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from waveformtools.models.lal import LALWaveformModel
from waveformtools.modes_array import ModesArray

try:
    from lalsimulation import SimInspiralGetApproximantFromString
except Exception:
    SimInspiralGetApproximantFromString = None

## Configuration

In [ ]:
TD_APPROXIMANT = "NRSur7dq4"
FD_APPROXIMANT = "IMRPhenomXPHM"

TD_F_LOWER = 0.0
FD_F_LOWER = 10.0

COMMON_PARAMS = dict(
    mass1=35.0,
    mass2=30.0,
    spin1x=0.0,
    spin1y=0.0,
    spin1z=0.2,
    spin2x=0.0,
    spin2y=0.0,
    spin2z=-0.1,
    distance=400.0,
    inclination=0.7,
    phi_ref=0.3,
    f_ref=20.0,
    f_max=512.0,
    delta_t=1.0 / 4096.0,
    delta_f=1.0 / 8.0,
    ell_max=4,
)

MODE_TO_PLOT = (2, 2)

## Helpers

In [ ]:
def make_params(approximant):
    params = dict(COMMON_PARAMS)
    params["approximant"] = approximant
    if approximant == TD_APPROXIMANT:
        params["f_lower"] = TD_F_LOWER
    elif approximant == FD_APPROXIMANT:
        params["f_lower"] = FD_F_LOWER
    else:
        raise ValueError(f"Unexpected approximant {approximant!r}")
    return params


def print_param_difference(td_params, fd_params):
    keys = sorted(set(td_params) | set(fd_params))
    print("Parameter comparison: TD vs FD")
    for key in keys:
        td_val = td_params.get(key)
        fd_val = fd_params.get(key)
        flag = "DIFF" if td_val != fd_val else "same"
        print(f"{key:>16s}: {td_val!r:>20}   {fd_val!r:>20}   {flag}")


def lal_string_is_available(approximant):
    if SimInspiralGetApproximantFromString is None:
        return False
    try:
        SimInspiralGetApproximantFromString(approximant)
        return True
    except Exception:
        return False


def summarize_modes(wfm, name, max_print=20):
    print(f"\n{name}")
    print("-" * len(name))
    print("type:", type(wfm))
    print("label:", getattr(wfm, "label", None))
    print("ell_max:", getattr(wfm, "ell_max", None))
    print("data_len:", getattr(wfm, "data_len", None))
    try:
        print("time range:", float(wfm.time_axis[0]), float(wfm.time_axis[-1]), "N=", len(wfm.time_axis))
    except Exception as exc:
        print("time range unavailable:", repr(exc))
    try:
        print("freq range:", float(wfm.frequency_axis[0]), float(wfm.frequency_axis[-1]), "N=", len(wfm.frequency_axis))
    except Exception as exc:
        print("freq range unavailable:", repr(exc))

    present = []
    missing = []
    for ell in range(2, int(wfm.ell_max) + 1):
        for m in range(-ell, ell + 1):
            try:
                data = wfm.mode(ell, m)
                if data is None:
                    missing.append((ell, m))
                else:
                    present.append((ell, m))
            except Exception:
                missing.append((ell, m))
    print(f"present modes ({len(present)}):", present[:max_print], "..." if len(present) > max_print else "")
    print(f"missing modes ({len(missing)}):", missing[:max_print], "..." if len(missing) > max_print else "")
    return present, missing

In [ ]:
def relative_l2(a, b, eps=1e-300):
    a = np.asarray(a)
    b = np.asarray(b)
    return np.linalg.norm(a - b) / max(np.linalg.norm(a), np.linalg.norm(b), eps)


def compare_common_modes(a, b, label_a="a", label_b="b", rtol=1e-10, atol=1e-12, max_print=20):
    common = []
    rows = []
    ell_max = min(int(a.ell_max), int(b.ell_max))
    for ell in range(2, ell_max + 1):
        for m in range(-ell, ell + 1):
            try:
                am = np.asarray(a.mode(ell, m))
                bm = np.asarray(b.mode(ell, m))
            except Exception:
                continue
            if am.shape != bm.shape:
                rows.append((ell, m, "shape_mismatch", am.shape, bm.shape, np.nan, np.nan))
                continue
            err_abs = np.max(np.abs(am - bm))
            err_rel = relative_l2(am, bm)
            ok = np.allclose(am, bm, rtol=rtol, atol=atol)
            rows.append((ell, m, ok, am.shape, bm.shape, float(err_abs), float(err_rel)))
            common.append((ell, m))
    failures = [r for r in rows if r[2] is not True]
    print(f"Compared {len(common)} common modes: {label_a} vs {label_b}")
    print(f"Failures / mismatches: {len(failures)}")
    for row in failures[:max_print]:
        print(row)
    return rows, failures


def peak_centered_time(wfm, mode=MODE_TO_PLOT):
    y = np.asarray(wfm.mode(*mode))
    t = np.asarray(wfm.time_axis)
    if len(t) != len(y):
        return np.arange(len(y))
    return t - t[int(np.argmax(np.abs(y)))]


def normalized_mode(wfm, mode=MODE_TO_PLOT):
    y = np.asarray(wfm.mode(*mode))
    scale = np.max(np.abs(y))
    if scale == 0 or not np.isfinite(scale):
        scale = 1.0
    return y / scale

In [ ]:
def plot_mode_real(waveforms, title, mode=MODE_TO_PLOT, normalize=False, peak_center=True, xlim=None):
    plt.figure(figsize=(8, 3.5))
    for label, wfm in waveforms:
        y = normalized_mode(wfm, mode=mode) if normalize else np.asarray(wfm.mode(*mode))
        x = peak_centered_time(wfm, mode=mode) if peak_center else np.asarray(wfm.time_axis)
        plt.plot(x, y.real, label=label)
    if xlim is not None:
        plt.xlim(*xlim)
    plt.xlabel("t - t_peak" if peak_center else "time")
    plt.ylabel(f"Re h_{{{mode[0]}{mode[1]}}}" + (" / max|h|" if normalize else ""))
    plt.title(title)
    plt.legend()
    plt.tight_layout()


def plot_mode_amp(waveforms, title, mode=MODE_TO_PLOT, normalize=True, peak_center=True, xlim=None):
    plt.figure(figsize=(8, 3.5))
    for label, wfm in waveforms:
        y = normalized_mode(wfm, mode=mode) if normalize else np.asarray(wfm.mode(*mode))
        x = peak_centered_time(wfm, mode=mode) if peak_center else np.asarray(wfm.time_axis)
        plt.plot(x, np.abs(y), label=label)
    if xlim is not None:
        plt.xlim(*xlim)
    plt.xlabel("t - t_peak" if peak_center else "time")
    plt.ylabel(f"|h_{{{mode[0]}{mode[1]}}}|" + (" / max|h|" if normalize else ""))
    plt.title(title)
    plt.legend()
    plt.tight_layout()


def plot_route_difference(reference, other, ref_label, other_label, mode=MODE_TO_PLOT):
    ref = np.asarray(reference.mode(*mode))
    oth = np.asarray(other.mode(*mode))
    n = min(len(ref), len(oth))
    diff = oth[:n] - ref[:n]
    t = np.asarray(reference.time_axis[:n])
    plt.figure(figsize=(8, 3.2))
    plt.plot(t, diff.real, label="Re difference")
    plt.plot(t, diff.imag, label="Im difference")
    plt.xlabel("time")
    plt.ylabel(f"{other_label} - {ref_label}")
    plt.title(f"FD→TD route difference for h_{{{mode[0]}{mode[1]}}}")
    plt.legend()
    plt.tight_layout()

## 1. Verify parameter consistency

In [ ]:
td_params = make_params(TD_APPROXIMANT)
fd_params = make_params(FD_APPROXIMANT)
print_param_difference(td_params, fd_params)

expected_differences = {"approximant", "f_lower"}
actual_differences = {key for key in sorted(set(td_params) | set(fd_params)) if td_params.get(key) != fd_params.get(key)}
assert actual_differences == expected_differences, actual_differences

## 2. Generate NRSur TD modes with `f_lower = 0`

In [ ]:
assert td_params["f_lower"] == 0.0
assert lal_string_is_available(TD_APPROXIMANT), f"{TD_APPROXIMANT} is not available in this LALSuite build"

td_model = LALWaveformModel(parameters_dict=td_params)
w_td_native = td_model.get_td_waveform_modes(dimensionless=False)
assert isinstance(w_td_native, ModesArray)
present_td, missing_td = summarize_modes(w_td_native, f"native TD modes: {TD_APPROXIMANT}, f_lower={TD_F_LOWER}")

## 3. Generate FD approximant modes with `f_lower = 10 Hz` and convert to TD

In [ ]:
assert fd_params["f_lower"] == 10.0
assert lal_string_is_available(FD_APPROXIMANT), f"{FD_APPROXIMANT} is not available in this LALSuite build"

fd_model = LALWaveformModel(parameters_dict=fd_params)
w_fd = fd_model.get_fd_waveform_modes(dimensionless=False)
w_fd_as_td = fd_model.get_fd_waveform_modes_as_td(dimensionless=False)
w_fd_as_td_alias = fd_model.get_fd_modes_as_td(dimensionless=False)
w_fd_auto_td = fd_model.get_td_waveform_modes(dimensionless=False)
w_fd_manual_as_td = w_fd.to_time_basis()

for w in [w_fd, w_fd_as_td, w_fd_as_td_alias, w_fd_auto_td, w_fd_manual_as_td]:
    assert isinstance(w, ModesArray)

present_fd, missing_fd = summarize_modes(w_fd, f"native FD modes: {FD_APPROXIMANT}, f_lower={FD_F_LOWER}")
present_fd_td, missing_fd_td = summarize_modes(w_fd_as_td, f"FD modes as TD: {FD_APPROXIMANT}, f_lower={FD_F_LOWER}")

## 4. Compare different FD→TD implementations/routes

In [ ]:
comparisons = [
    (w_fd_as_td, w_fd_as_td_alias, "explicit FD→TD", "alias FD→TD"),
    (w_fd_as_td, w_fd_auto_td, "explicit FD→TD", "automatic get_td_waveform_modes"),
    (w_fd_as_td, w_fd_manual_as_td, "explicit FD→TD", "manual FD.to_time_basis"),
]
all_failures = {}
for a, b, la, lb in comparisons:
    rows, failures = compare_common_modes(a, b, label_a=la, label_b=lb, rtol=0.0, atol=0.0)
    all_failures[(la, lb)] = failures

assert len(all_failures[("explicit FD→TD", "alias FD→TD")]) == 0
assert len(all_failures[("explicit FD→TD", "automatic get_td_waveform_modes")]) == 0
assert len(all_failures[("explicit FD→TD", "manual FD.to_time_basis")]) == 0

## 5. Dimensionless FD→TD convention check

In [ ]:
w_fd_auto_td_dimless = fd_model.get_td_waveform_modes(dimensionless=True)
w_fd_as_td_dimless = fd_model.get_fd_waveform_modes_as_td(dimensionless=True)
rows, failures = compare_common_modes(
    w_fd_auto_td_dimless,
    w_fd_as_td_dimless,
    label_a="dimensionless automatic get_td_waveform_modes",
    label_b="dimensionless explicit FD→TD",
    rtol=0.0,
    atol=0.0,
)
assert len(failures) == 0

## 6. Visual check: NRSur TD waveform and FD→TD waveform

This is a qualitative overlay of different approximants with consistent shared parameters, except for the requested `f_lower` choices.

In [ ]:
plot_mode_real(
    [
        (f"{TD_APPROXIMANT}, f_lower={TD_F_LOWER}", w_td_native),
        (f"{FD_APPROXIMANT} FD→TD, f_lower={FD_F_LOWER}", w_fd_as_td),
    ],
    title="TD approximant vs FD→TD approximant: Re h22",
    normalize=True,
    peak_center=True,
    xlim=(-500, 100),
)
plot_mode_amp(
    [
        (f"{TD_APPROXIMANT}, f_lower={TD_F_LOWER}", w_td_native),
        (f"{FD_APPROXIMANT} FD→TD, f_lower={FD_F_LOWER}", w_fd_as_td),
    ],
    title="TD approximant vs FD→TD approximant: |h22|",
    normalize=True,
    peak_center=True,
    xlim=(-500, 100),
)

## 7. Visual check: FD→TD route overlays

In [ ]:
fd_td_routes = [
    ("explicit get_fd_waveform_modes_as_td", w_fd_as_td),
    ("alias get_fd_modes_as_td", w_fd_as_td_alias),
    ("automatic get_td_waveform_modes", w_fd_auto_td),
    ("manual FD.to_time_basis", w_fd_manual_as_td),
]
plot_mode_real(fd_td_routes, title=f"FD→TD route comparison: {FD_APPROXIMANT}", normalize=False, peak_center=True)
plot_mode_amp(fd_td_routes, title=f"FD→TD route amplitude comparison: {FD_APPROXIMANT}", normalize=True, peak_center=True)

## 8. Difference plots between FD→TD routes

These should be zero or near numerical roundoff if the implementations are equivalent.

In [ ]:
plot_route_difference(w_fd_as_td, w_fd_as_td_alias, "explicit", "alias")
plot_route_difference(w_fd_as_td, w_fd_auto_td, "explicit", "automatic")
plot_route_difference(w_fd_as_td, w_fd_manual_as_td, "explicit", "manual")